# Tome 7, Chapter 4: The Spectral Theorem (Python companion)

Exercises first, solutions at the end. The spectral theorem on a computer: spectral clustering, principal components of Brownian motion, listening to a string, and how smoothness of a kernel controls its eigenvalues. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★★) CS application: spectral clustering.** Generate two interleaved half-moons of $300$ points in the plane with a little noise. Build the similarity graph with weights $w_{ij}=e^{-|x_i-x_j|^2/(2\sigma^2)}$, its Laplacian $L=D-W$ (Exercise 4.4), and split the points by the sign of the eigenvector of the second smallest eigenvalue. Compare with $k$-means on the raw coordinates. Why does the Laplacian see the moons while $k$-means does not?

**P4.2 (★) Principal components of Brownian motion.** Simulate $5000$ Brownian paths on a grid of $200$ points of $[0,1]$, compute the empirical covariance matrix and its eigendecomposition, and compare the eigenvalues and eigenvectors with Example 4.15. What fraction of the variance do the first $1$, $3$ and $10$ modes carry? Rebuild one path from its first $N$ modes.

**P4.3 (★) Listening to a string.** Discretise $-u''+qu=\lambda u$ on $[0,1]$ with Dirichlet conditions by finite differences, for $q=0$ and $q(x)=60x$. Check the bounds $\pi^2n^2\le\lambda_n\le\pi^2n^2+\max q$ of Theorem 4.11, plot $N(\lambda)$ against $\frac{\sqrt\lambda}\pi$, and recover the length $\ell$ of a string on $[0,\ell]$ from its first $50$ eigenvalues. When do the finite-difference eigenvalues stop being trustworthy?

**P4.4 (★) Smooth kernels, fast decay.** Compute (by the Nyström method of the Chapter 3 notebook) the eigenvalues of the integral operators on $[0,1]$ with kernels $\min(s,t)$, $e^{-|s-t|}$ and $e^{-(s-t)^2/(2\cdot0.2^2)}$. Check Mercer's trace formula for each, and compare the decay rates of the eigenvalues. What does this say about how well each kernel matrix can be approximated by a matrix of low rank?

## Solutions

### P4.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
n = 150
t = rng.uniform(0, np.pi, n)
A = np.c_[np.cos(t), np.sin(t)]
B = np.c_[1 - np.cos(t), 0.5 - np.sin(t)]
X = np.r_[A, B] + 0.06 * rng.standard_normal((2 * n, 2))
truth = np.r_[np.zeros(n), np.ones(n)]

def kmeans(X, k=2, iters=100, seed=0):
    r = np.random.default_rng(seed)
    C = X[r.choice(len(X), k, replace=False)]
    for _ in range(iters):
        lab = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
        C = np.array([X[lab == j].mean(0) for j in range(k)])
    return lab

def accuracy(lab):
    return max(np.mean(lab == truth), np.mean(lab != truth))

sigma = 0.1
D2 = ((X[:, None, :] - X[None]) ** 2).sum(-1)
W = np.exp(-D2 / (2 * sigma ** 2)); np.fill_diagonal(W, 0)
L = np.diag(W.sum(1)) - W
mu, F = np.linalg.eigh(L)
fiedler = F[:, 1]
lab_spec = (fiedler > 0).astype(int)
lab_km = kmeans(X)
print("smallest eigenvalues of L:", np.round(mu[:4], 4))
print(f"spectral clustering accuracy: {accuracy(lab_spec):.3f}")
print(f"k-means accuracy:             {accuracy(lab_km):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].scatter(*X.T, c=lab_km, s=8, cmap='coolwarm'); ax[0].set_title('k-means')
ax[1].scatter(*X.T, c=lab_spec, s=8, cmap='coolwarm'); ax[1].set_title('sign of the Fiedler vector')
plt.show()

$k$-means cuts the plane by a straight line (the perpendicular bisector of its two centres), so it must slice through the interleaved moons and gets only about three quarters of the points right. The Laplacian sees the *graph*: with $\sigma=0.1$, points are strongly linked only to close neighbours, each moon is a long, well-connected chain, and the two moons are linked by very few weak edges. The second eigenvalue is tiny (the graph is nearly disconnected) and its eigenvector is nearly constant on each moon, with opposite signs (Exercise 4.4(b): it minimises $\sum w_{ij}(x_i-x_j)^2$ among vectors orthogonal to the constants). Its sign recovers the moons exactly here. The method depends on $\sigma$: too small and the graph falls apart into many pieces, too large and it becomes the complete graph, where the Laplacian sees only distances again. In practice one uses $k$ nearest neighbours and the normalised Laplacian $I-D^{-1/2}WD^{-1/2}$ (Shi–Malik, Ng–Jordan–Weiss).

### P4.2

In [ ]:
m, paths = 200, 5000
t = np.arange(1, m + 1) / m
dB = rng.standard_normal((paths, m)) / np.sqrt(m)
Bm = np.cumsum(dB, axis=1)
C = Bm.T @ Bm / paths                         # empirical covariance, close to min(s,t)
w, V = np.linalg.eigh(C / m)                  # the factor 1/m turns the matrix into the integral operator
w, V = w[::-1], V[:, ::-1] * np.sqrt(m)       # eigenvectors normalised in L^2([0,1])
lam = 1 / ((np.arange(1, 11) - 0.5) ** 2 * np.pi ** 2)
for k in range(5):
    e = np.sqrt(2) * np.sin((k + 0.5) * np.pi * t)
    sign = np.sign(V[:, k] @ e)
    print(f"mode {k+1}: empirical {w[k]:.5f}  theory {lam[k]:.5f}   L2 distance of eigenvectors {np.sqrt(np.mean((sign * V[:, k] - e) ** 2)):.3f}")
total = w.sum()
for N in [1, 3, 10]:
    print(f"first {N:2d} modes carry {w[:N].sum() / total:.1%} of the variance (theory: {lam[:N].sum() / 0.5:.1%})")
b = Bm[0]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t, b, 'k', lw=0.8, label='path')
for N in [3, 10, 50]:
    coeffs = V[:, :N].T @ b / m
    ax.plot(t, V[:, :N] @ coeffs, label=f'{N} modes')
ax.legend(fontsize=8); plt.show()

The eigenvalues of the empirical covariance operator match $\frac1{(n-\frac12)^2\pi^2}$ to within a few percent (the sampling error of $5000$ paths), and the eigenvectors are the sine waves $\sqrt2\sin((n-\frac12)\pi t)$ of Example 4.15, up to sign and small noise. The first mode carries about $81\%$ of the variance ($\frac8{\pi^2}$), three modes about $93\%$, and ten modes about $98\%$: Brownian motion is mostly a single slow wave. This is principal component analysis of a random function, and it is the same computation as PCA of a data matrix: diagonalise the covariance. The reconstruction shows the trade-off: a few modes give the shape, and the roughness lives in the long tail of modes with variances decreasing only like $\frac1{n^2}$, which is why Brownian paths are continuous but nowhere differentiable.

### P4.3

In [ ]:
def sl_eigs(qfun, n=2000, ell=1.0):
    h = ell / (n + 1); x = np.linspace(h, ell - h, n)
    A = (np.diag(2 * np.ones(n)) - np.diag(np.ones(n - 1), 1) - np.diag(np.ones(n - 1), -1)) / h ** 2 + np.diag(qfun(x))
    return np.linalg.eigvalsh(A)
l0 = sl_eigs(lambda x: 0 * x)
lq = sl_eigs(lambda x: 60 * x)
k = np.arange(1, len(l0) + 1)
for n in [1, 2, 5, 10, 50, 200, 1000]:
    print(f"n={n:4d}: q=0 {l0[n-1]:12.2f} (pi^2 n^2 = {np.pi**2 * n**2:12.2f})   q=60x {lq[n-1]:12.2f}   difference {lq[n-1] - l0[n-1]:6.2f}")
lam = np.linspace(1, 4e4, 400)
plt.plot(np.sqrt(lam) / np.pi, [np.sum(lq <= L) for L in lam], label='N(lambda), q = 60x')
plt.plot(np.sqrt(lam) / np.pi, np.sqrt(lam) / np.pi, 'k--', label='sqrt(lambda)/pi')
plt.xlabel('sqrt(lambda)/pi'); plt.legend(); plt.show()
ell = 2.7
ev = sl_eigs(lambda x: 0 * x, ell=ell)[:50]
print("length recovered from 50 eigenvalues:", np.mean(np.pi * np.arange(1, 51) / np.sqrt(ev)))

For the low modes, the finite-difference eigenvalues match $\pi^2n^2$ to several digits, and with $q=60x$ the shift $\lambda_n(q)-\lambda_n(0)$ lies between $0$ and $60$, as Theorem 4.11 says. In fact it tends to $\int_0^1q=30$: a high mode oscillates so fast that it only feels the average of the potential (a first glimpse of first-order perturbation theory, $\langle qe_n,e_n\rangle\approx\int q$ since $e_n^2=2\sin^2(n\pi x)$ averages to $1$). The counting function follows $\frac{\sqrt\lambda}\pi$, Weyl's law, and the length of a string of length $2.7$ is recovered from $\pi n/\sqrt{\lambda_n}$. But at $n=1000$ the discrete eigenvalue is far below $\pi^2n^2$: the matrix eigenvalues are $\frac4{h^2}\sin^2\frac{n\pi h}2$, and once the wavelength approaches the grid step the discretisation cannot represent the mode. With $2000$ grid points the eigenvalues are accurate to $1\%$ only up to about $n=200$, that is with some twenty points per wavelength: a rule of thumb every numerical analyst learns.

### P4.4

In [ ]:
from numpy.polynomial.legendre import leggauss
def nystrom_eigs(kernel, n=300):
    s, w = leggauss(n); s = (s + 1) / 2; w = w / 2
    K = kernel(s[:, None], s[None, :])
    M = np.sqrt(w)[:, None] * K * np.sqrt(w)[None, :]   # symmetric version of K diag(w)
    return np.sort(np.linalg.eigvalsh(M))[::-1], np.sum(w * kernel(s, s))
kernels = {"min(s,t)": lambda s, t: np.minimum(s, t),
           "exp(-|s-t|)": lambda s, t: np.exp(-np.abs(s - t)),
           "Gaussian, width 0.2": lambda s, t: np.exp(-(s - t) ** 2 / (2 * 0.2 ** 2))}
for name, k in kernels.items():
    ev, tr = nystrom_eigs(k)
    print(f"{name:20s}: sum of eigenvalues {ev.sum():.6f}, integral of the diagonal {tr:.6f}")
    print("    eigenvalues 1, 2, 5, 10, 20:", " ".join(f"{ev[i]:.2e}" for i in [0, 1, 4, 9, 19]))
    plt.semilogy(np.arange(1, 41), np.abs(ev[:40]), 'o-', ms=3, label=name)
plt.xlabel('n'); plt.ylabel('lambda_n'); plt.legend(); plt.show()

Mercer's trace formula holds to many digits in all three cases ($\frac12$, $1$ and $1$). The decay rates differ completely. For $\min(s,t)$ and $e^{-|s-t|}$, both with a corner on the diagonal, the eigenvalues decay like $\frac1{n^2}$; for the Gaussian kernel, which is analytic, they decay faster than any exponential, and after about twenty of them they reach rounding level (printed values near $10^{-16}$ are noise and may even be slightly negative). This is a general principle: the smoother the kernel, the faster its eigenvalues decay. In machine learning it decides whether a kernel matrix $K_{ij}=k(x_i,x_j)$ on $10^6$ points can be replaced by a matrix of rank $50$ (the Nyström approximation and random features), and by Theorem 4.6 the best rank-$r$ approximation error in operator norm is exactly $\lambda_{r+1}$. For a Gaussian kernel, rank $20$ is as good as the full matrix; for the Brownian kernel, rank $r$ leaves an error of order $\frac1{r^2}$.